### `etd_tests.ipynb` 
*Created: Sept 22, 2026* <br/>
Notebook for testing some Exponential Time Differencing (ETD) solvers by comparing them against trusted reference solvers from `OrdinaryDiffEq.jl`. We also test against exact solutions, when available.

In [1]:
using OrdinaryDiffEq, CairoMakie, NBInclude, UnPack, Printf, Test, LinearAlgebra, LaTeXStrings, Statistics
import OrdinaryDiffEqCore: OrdinaryDiffEqAlgorithm  

In [9]:
using NBInclude
@nbinclude("etd_euler.ipynb")
@nbinclude("etd_rk2.ipynb")
@nbinclude("etd_rk3.ipynb")
@nbinclude("etd_rk4.ipynb")
@nbinclude("../../../../set_makie_defaults.ipynb")
#@nbinclude("plotting_utils.ipynb")

#Import solvers
@nbinclude("etd/etd_euler.ipynb")

#Import ODE test problems
@nbinclude("../../ode_library.ipynb")

In [ ]:
struct SemilinearTestProblem{M, F, U, P, S}
    A::M
    f::F
    u0::U
    tspan::NTuple{2,Float64}
    p::P
    exact_solution::S
end 

function SemilinearTestProblem(A::M, f::F, u0::U, tspan::NTuple{2,<:Real}, p::P = nothing; exact_solution::S = nothing) where {M,F,U,P,S}
    tspan = Float64.(tspan)
    return ODETestProblem(A, f, u0, tspan, p, exact_solution)
end

In [10]:
discrete_lp_norm(u, Δt; p = 2)  = norm(u,p) * (Δt)^(1/p)

discrete_lp_norm (generic function with 1 method)

### Scalar Test ODE: 

##### **ODE: $\displaystyle u' = -au + u^2, \quad u(0) = u_0, \quad 0 < u_0 < a$**

##### **Exact Solution:**  $\quad \displaystyle u(t) = \frac{a}{1 - \left(1 - \frac{a}{u_0} \right)e^t}$

- We have $A = -a$ and $f(u,t) = u^2$
- Note that $\lim\limits_{t \to \infty} u(t) = 0$, provided $a > 0$. 

In [11]:
#Test ETD Euler using a *scalar* equation
a = 3.0 
u0 = 1.0 

Δt = 0.01
tf = 2.0 
u_exact_scalar(t,a,u0) = a ./ (1.0 - (1.0 - a/u0)*exp(a*t))

A = -a 
f(u,p,t) = u^2 

for solver in [etd_euler, etd_rk2, etd_rk3, etd_rk4]

    sol = solver(A, f, u0; Δt = 0.1, tspan = (0,tf), p = nothing, ϵ = 1e-3)
    
    @unpack u, t, Δt = sol
    u_num = u 
    u_ex = u_exact_scalar.(t, a, u0)
    
    title_str = string(nameof(solver)) * @sprintf("    MSE = %.4e", mean((u_num .- u_ex).^2))
    plot_sol_1D(u_num, t; u_ex, size = (400,400), title = title_str, markersize = 4, linewidth = 4, legend_pos = :rt) 
end

LoadError: UndefVarError: `plot_sol_1D` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

#### **Vector Test ODE** 
$$A = \begin{pmatrix} -100 & \phantom{-}0 \\ 0 & -1 \end{pmatrix}, \quad F(\mathbf{u},p,t) = \begin{pmatrix} \cos t + 100 \sin t + u_1^2 - \sin^2 t \\ - \sin t + \cos t + u_1 u_2 - \sin t \cos t \end{pmatrix}, \quad \mathbf{u}(0) = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

#### **Exact solution:** $$\mathbf{u}(t) = \begin{pmatrix} u_1(t) \\ u_2(t) \end{pmatrix} = \begin{pmatrix} \sin t \\ \cos t \end{pmatrix}$$



In [ ]:
function f(u,p,t)
    f1 = cos(t) + 100*sin(t) + (u[1])^2 - (sin(t))^2 
    f2 = -sin(t) + cos(t) + u[1] * u[2] - sin(t) * cos(t)
    return [f1, f2]
end 

u_exact(t) = [sin(t), cos(t)]

A = [-100.0 0.0; 0.0 -1.0]
u0 = [0.0, 1.0]

#Test ETD Euler using a *scalar* equation
for solver in [etd_euler, etd_rk2, etd_rk3, etd_rk4]

    sol = solver(A, f, u0; Δt = 0.0001, tspan = (0,tf), p = nothing, ϵ = 1e-3)

    t = sol.t
    U_num = sol.u
    U_exact = u_exact.(t)

    l2_errors = [norm(U_num[i] - U_exact[i]) for i=1:length(U_num)]
    mean_l2_err = mean(l2_errors)
    
    title_str = string(nameof(solver)) * @sprintf("    Mean L2 Error = %.6e", mean_l2_err)
    plot_sol_2D(U_num, t; U_exact = U_exact, size = (400,400),  title = title_str, markersize = 4, linewidth = 4)
end

In [ ]:
function compare_etd_solutions(prob::ODETestProblem, reference_alg::OrdinaryDiffEqAlgorithm, 
                               custom_alg::A; dt::Real = 0.01) where {A}
    """    
    reference_alg :: algorithm from the OrdinaryDiffEq package 
    custom_alg :: algorithm that I wrote, that I'm testing 
  
    Valid options for `custom_alg`: 
        - euler
        - rk4

        ...will add more later....
    """
  
    @unpack f, u0, tspan, p = prob

    #STEP 1: Solve the ODE using the custom algorithm 
    custom_sol = custom_alg(f, u0, tspan, p; dt = dt)

    #STEP 2: Solve the ODE using the reference algorithm (an algorithm from OrdinaryDiffEq.jl)
    reference_sol = solve(ODEProblem(f, u0, tspan, p), reference_alg; adaptive = false, dt = dt)

    #Ensure that each algorithm is solving the problem at the time values (a very small tolerance is permitted)
    t_diff = maximum(custom_sol.t .- reference_sol.t)
    t_tol = 1e-10
    
    if t_diff > t_tol
        throw(error("custom_sol.t and reference_sol.t differ by more than " * @sprintf("%.4e", t_tol)))
    end 
    
    #STEP 3: Compute difference between reference solution and custom solution 
    u_custom = custom_sol.u
    u_reference = reference_sol.u
    max_l2_error = maximum(norm.(u_reference .- u_custom))

    return (reference_sol = reference_sol, custom_sol = custom_sol, max_l2_error = max_l2_error)     
end 